In [3]:
import pandas as pd
from sklearn import tree
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import matplotlib.pyplot as plt

ModuleNotFoundError: No module named 'sklearn'

In [2]:
df_titanic = pd.read_csv('../../datasets/structured/titanic.csv')
import os
os.getcwd()
df_titanic.head()

NameError: name 'pd' is not defined

In [42]:
df_titanic.columns

Index(['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp',
       'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked'],
      dtype='object')

In [43]:
df_titanic.describe()

,PassengerId,Survived,Pclass,Age,SibSp,Parch,Fare
count,891.000000,891.000000,891.000000,714.000000,891.000000,891.000000,891.000000
mean,446.000000,0.383838,2.308642,29.699118,0.523008,0.381594,32.204208
std,257.353842,0.486592,0.836071,14.526497,1.102743,0.806057,49.693429
min,1.000000,0.000000,1.000000,0.420000,0.000000,0.000000,0.000000
25%,223.500000,0.000000,2.000000,20.125000,0.000000,0.000000,7.910400
50%,446.000000,0.000000,3.000000,28.000000,0.000000,0.000000,14.454200
75%,668.500000,1.000000,3.000000,38.000000,1.000000,0.000000,31.000000
max,891.000000,1.000000,3.000000,80.000000,8.000000,6.000000,512.329200


In [44]:
df_titanic.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB


In [45]:
df_titanic.drop(["PassengerId", "Name", "Ticket", "Cabin","Fare","Parch"], axis=1, inplace=True, errors='ignore')
df_titanic = df_titanic.dropna(subset=['Embarked'])
mean_age = df_titanic['Age'].mean()
df_titanic['Age'].fillna(mean_age, inplace=True)
df_titanic.columns

/tmp/ipython-input-321590277.py:4: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_titanic['Age'].fillna(mean_age, inplace=True)


Index(['Survived', 'Pclass', 'Sex', 'Age', 'SibSp', 'Embarked'], dtype='object')

In [46]:
df_titanic.info()

<class 'pandas.core.frame.DataFrame'>
Index: 889 entries, 0 to 890
Data columns (total 6 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Survived  889 non-null    int64  
 1   Pclass    889 non-null    int64  
 2   Sex       889 non-null    object 
 3   Age       889 non-null    float64
 4   SibSp     889 non-null    int64  
 5   Embarked  889 non-null    object 
dtypes: float64(1), int64(3), object(2)
memory usage: 48.6+ KB


In [47]:
# แปลงข้อมูลข้อความเป็นตัวเลข
df_titanic["Sex"] = df_titanic["Sex"].map({"male": 0, "female": 1})

if "Embarked" in df_titanic.columns:
    df_titanic["Embarked"] = df_titanic["Embarked"].map({"S": 0, "C": 1, "Q": 2})


In [48]:
df_titanic.fillna(df_titanic.median(numeric_only=True), inplace=True)

In [49]:
y = df_titanic["Survived"].values
X = df_titanic.drop(["Survived"], axis=1)

In [50]:
from sklearn.preprocessing import StandardScaler #z-score
#from sklearn.preprocessing import MixMaxScaler

#ทำ Data Normalization ด้วย StandardScaler
scaler = StandardScaler()
#scaler = MixMaxScaler()
scaler.fit(X)
X = scaler.transform(X)
X = pd.DataFrame(X, columns=df_titanic.columns[1:])
X.head()

,Pclass,Sex,Age,SibSp,Embarked
0,0.825209,-0.735342,-0.589620,0.431350,-0.569684
1,-1.572211,1.359911,0.644848,0.431350,1.003139
2,0.825209,1.359911,-0.281003,-0.475199,-0.569684
3,-1.572211,1.359911,0.413385,0.431350,-0.569684
4,0.825209,-0.735342,0.413385,-0.475199,-0.569684


In [51]:
from sklearn.model_selection import StratifiedKFold
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split
import numpy as np

kf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

k_values = [3, 5, 7, 9]   # ✅ ทดสอบเฉพาะ k = 3, 5, 7, 9
mean_accuracies = []

for k in k_values:
    fold_acc = []
    for train_idx, test_idx in kf.split(X, y):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        # สร้างโมเดล KNN โดยกำหนด k เพื่อนบ้าน
        clf = KNeighborsClassifier(n_neighbors=k)
        clf.fit(X_train, y_train)
        y_pred = clf.predict(X_test)

        # เก็บ Accuracy ของแต่ละ fold
        acc = accuracy_score(y_test, y_pred)
        fold_acc.append(acc)

    # เก็บค่าเฉลี่ยของ accuracy ของแต่ละค่า k
    mean_accuracies.append(np.mean(fold_acc))

best_k = k_values[np.argmax(mean_accuracies)]
best_acc = max(mean_accuracies)
clf.fit(X_train, y_train )

KNeighborsClassifier(n_neighbors=9)

In [52]:
df_titanic_predict = X_test.copy()
df_titanic_predict['predicted'] = clf.predict(X_test)
df_titanic_predict['actual'] = y_test

target_names = {0: 'Not Survived', 1: 'Survived'}
df_titanic_predict['actual'] = df_titanic_predict['actual'].map(target_names)
df_titanic_predict['predicted'] = df_titanic_predict['predicted'].map(target_names)

df_titanic_predict.head(10)

,Pclass,Sex,Age,SibSp,Embarked,predicted,actual
1,-1.572211,1.359911,6.448480e-01,0.431350,1.003139,Survived,Survived
3,-1.572211,1.359911,4.133853e-01,0.431350,-0.569684,Survived,Survived
5,0.825209,-0.735342,-5.482138e-16,-0.475199,2.575963,Not Survived,Not Survived
7,0.825209,-0.735342,-2.132705e+00,2.244449,-0.569684,Not Survived,Not Survived
8,0.825209,1.359911,-2.038487e-01,-0.475199,-0.569684,Survived,Survived
10,0.825209,1.359911,-1.978396e+00,0.431350,-0.569684,Not Survived,Survived
11,-1.572211,1.359911,2.187933e+00,-0.475199,-0.569684,Survived,Survived
12,0.825209,-0.735342,-7.439283e-01,-0.475199,-0.569684,Not Survived,Not Survived
14,0.825209,1.359911,-1.206854e+00,-0.475199,-0.569684,Survived,Not Survived
15,-0.373501,1.359911,1.956470e+00,-0.475199,-0.569684,Survived,Survived


In [53]:
from sklearn import metrics
print(metrics.classification_report(y_test, clf.predict(X_test)))

              precision    recall  f1-score   support

           0       0.83      0.90      0.87       183
           1       0.82      0.71      0.76       113

    accuracy                           0.83       296
   macro avg       0.82      0.80      0.81       296
weighted avg       0.83      0.83      0.82       296



In [54]:
metrics.confusion_matrix(y_test, clf.predict(X_test))

array([[165,  18],
       [ 33,  80]])

In [55]:
clf.score(X_test, y_test)

0.8277027027027027

In [56]:
from sklearn.datasets import load_iris
from sklearn.model_selection import StratifiedKFold
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import numpy as np

k = 3

# Initialize Stratified KFold cross-validation
kf = StratifiedKFold(n_splits=k, shuffle=True, random_state=42)

# Initialize empty lists to store predictions and ground truth for all folds
all_y_true = []
all_y_pred = []
accuracy_scores=[]
confusion_matrices=[]

# Loop through each fold
for train_index, test_index in kf.split(X,y):
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y[train_index], y[test_index]

    # Create a decision tree classifier
    clf = KNeighborsClassifier(n_neighbors=9)

    # Fit the model to the training data
    clf.fit(X_train, y_train)

    # Make predictions on the test data
    y_pred = clf.predict(X_test)

    # Calculate accuracy for this fold and store it
    accuracy = accuracy_score(y_test, y_pred)
    accuracy_scores.append(accuracy)

    # Calculate confusion matrix and store it
    confusion = confusion_matrix(y_test, y_pred)
    confusion_matrices.append(confusion)

    # Store ground truth and predictions for this fold
    all_y_true.extend(y_test)
    all_y_pred.extend(y_pred)


# Calculate mean and standard deviation of accuracy scores
mean_accuracy = np.mean(accuracy_scores)
std_accuracy = np.std(accuracy_scores)

# Print evaluation metrics summary
print(f"Mean Accuracy: {mean_accuracy:.2f}")
print(f"Standard Deviation of Accuracy: {std_accuracy:.2f}")

# Print average confusion matrix (you can also compute other statistics from it)
sum_confusion_matrix = np.sum(confusion_matrices, axis=0)
print("\n\nSum Confusion Matrix:")
print(sum_confusion_matrix)


# Generate a classification report for all folds and classes
report = classification_report(all_y_true, all_y_pred, target_names=['Not Survived','Survived'])

# Display the classification report
print("\n\nClassification Report - All Folds and Classes:")
print(report)

Mean Accuracy: 0.82
Standard Deviation of Accuracy: 0.01


Sum Confusion Matrix:
[[499  50]
 [112 228]]


Classification Report - All Folds and Classes:
              precision    recall  f1-score   support

Not Survived       0.82      0.91      0.86       549
    Survived       0.82      0.67      0.74       340

    accuracy                           0.82       889
   macro avg       0.82      0.79      0.80       889
weighted avg       0.82      0.82      0.81       889

